<a href="https://colab.research.google.com/github/jana-nf/threat_infostealer_login_SQLInjection/blob/main/threat_correlator_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Threat Correlator: Infostealer → Login legítimo → SQL Injection

Simulação de um motor de correlação (estilo mini-SIEM) que liga três eventos que, isolados,
parecem pouco graves, mas juntos formam uma **cadeia de ataque**, e envia **alertas imediatos ao Telegram**.

> ⚠️ Tudo é **simulado**: eventos de log sintéticos, IPs de documentação (RFC 5737) e domínio fictício.
> Não há malware, exploit nem ataque real contra nenhum sistema.

| Estágio | Evento | Sinal |
|---|---|---|
| 1 | `credential_leak` | credencial corporativa apareceu em log de infostealer / fórum |
| 2 | `login` | login bem-sucedido com a conta vazada, de IP nunca visto (ou país fora do padrão) |
| 3 | `http_request` | requisição com padrão de SQL Injection vindo do mesmo IP/sessão do login |

**Severidade escalonada:** 1 → 🟡 médio · 1+2 → 🟠 alto · 1+2+3 → 🔴 crítico · SQLi com resposta grande → 🔴 possível exfiltração.

### Como usar
1. Alerta de mensagem para o Telegram. Sem isso, os alertas aparecem só na tela (modo DRY-RUN).
2. Menu **Ambiente de execução ▸ Executar tudo** (*Runtime ▸ Run all*).

## 1. Modelos de dados

In [1]:
"""Modelos de dados compartilhados: eventos de entrada e alertas de saída."""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum, IntEnum
from typing import Any


class EventType(str, Enum):
    CREDENTIAL_LEAK = "credential_leak"  # feed de threat intel (infostealer / fóruns)
    LOGIN = "login"                      # log de autenticação (VPN / SSO / AD)
    HTTP_REQUEST = "http_request"        # log de aplicação web / WAF


class Severity(IntEnum):
    MEDIUM = 1
    HIGH = 2
    CRITICAL = 3

    @property
    def label(self) -> str:
        return {1: "MÉDIO", 2: "ALTO", 3: "CRÍTICO"}[self.value]

    @property
    def emoji(self) -> str:
        return {1: "🟡", 2: "🟠", 3: "🔴"}[self.value]


# Estágios da kill chain monitorada
STAGE_NAMES = {
    1: "Credencial vazada (Infostealer)",
    2: "Login com credencial legítima",
    3: "SQL Injection",
}


@dataclass(frozen=True)
class Event:
    """Evento normalizado. Qualquer fonte (logs, APIs, SIEM) deve virar um Event."""

    type: EventType
    timestamp: datetime
    user: str | None = None
    src_ip: str | None = None
    country: str | None = None
    data: dict[str, Any] = field(default_factory=dict)

    def summary(self) -> str:
        parts = [f"[{self.type.value}]"]
        if self.user:
            parts.append(f"user={self.user}")
        if self.src_ip:
            parts.append(f"ip={self.src_ip}")
        if self.country:
            parts.append(f"país={self.country}")
        if self.type is EventType.HTTP_REQUEST:
            parts.append(f"{self.data.get('path', '')}?{self.data.get('query', '')}"[:90])
        return " ".join(parts)


@dataclass
class Alert:
    severity: Severity
    title: str
    stages: list[int]
    user: str | None = None
    src_ip: str | None = None
    country: str | None = None
    details: dict[str, str] = field(default_factory=dict)
    timeline: list[str] = field(default_factory=list)
    recommendations: list[str] = field(default_factory=list)
    created_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

## 2. Configuração

In [2]:
"""Configuração do motor de correlação."""
from __future__ import annotations

from dataclasses import dataclass
from datetime import timedelta


@dataclass(frozen=True)
class Config:
    telegram_token: str | None = None
    telegram_chat_id: str | None = None
    # Por quanto tempo uma credencial vazada é considerada "ativa" (até o reset de senha)
    leak_window: timedelta = timedelta(days=30)
    # Tempo máximo entre o login suspeito e o SQLi para serem considerados a mesma cadeia
    chain_window: timedelta = timedelta(hours=2)
    # Países de onde os logins são esperados (ajuste para a sua empresa)
    trusted_countries: frozenset = frozenset({"BR"})
    # Resposta HTTP 200 maior que isso, durante um SQLi, indica provável extração de dados
    exfil_bytes_threshold: int = 500_000
    # Evita spam: intervalo mínimo entre alertas de SQLi "solto" do mesmo IP
    standalone_cooldown: timedelta = timedelta(minutes=1)

## 3. Detector de SQL Injection
Assinaturas simples (estilo WAF): decodifica URL, dupla codificação e comentários inline.

In [3]:
"""Detecção defensiva de SQL Injection por assinaturas em requisições HTTP.

Esta é uma camada simples (estilo WAF). Em produção, ela seria complementada
por um WAF real (ModSecurity, Cloudflare, AWS WAF) e a correlação consumiria
os alertas dele.
"""
from __future__ import annotations

import re
from dataclasses import dataclass
from urllib.parse import unquote_plus


@dataclass(frozen=True)
class SqliResult:
    detected: bool
    signatures: tuple = ()
    score: int = 0


# (nome, peso, regex). Detecta se a soma dos pesos for >= SCORE_THRESHOLD.
_SIGNATURES = (
    ("UNION SELECT", 3, re.compile(r"\bunion\b(?:\s+all)?\s+select\b", re.I)),
    ("tautologia (OR/AND x=x)", 2,
     re.compile(r"\b(?:or|and)\b\s*\(?\s*['\"]?(\w+)['\"]?\s*=\s*['\"]?\1\b", re.I)),
    ("comando empilhado (;)", 3,
     re.compile(r";\s*(?:drop|insert|update|delete|alter|create|exec(?:ute)?|shutdown)\b", re.I)),
    ("blind/time-based", 3,
     re.compile(r"\b(?:sleep|benchmark|pg_sleep)\s*\(|\bwaitfor\s+delay\b", re.I)),
    ("enumeração de schema", 3,
     re.compile(r"\b(?:information_schema|sysobjects|pg_catalog|sqlite_master)\b", re.I)),
    ("função de erro/arquivo", 3,
     re.compile(r"\b(?:extractvalue|updatexml|load_file|xp_cmdshell)\b|\binto\s+(?:out|dump)file\b", re.I)),
    ("truncamento por comentário", 1, re.compile(r"['\"]\s*(?:--|#|/\*)")),
)
SCORE_THRESHOLD = 2

_INLINE_COMMENT = re.compile(r"/\*.*?\*/")


def _normalize(*parts: str) -> str:
    """Junta e decodifica URL (inclusive dupla codificação, ex.: %2527 -> %27 -> ')."""
    text = " ".join(p for p in parts if p)
    for _ in range(2):
        decoded = unquote_plus(text)
        if decoded == text:
            break
        text = decoded
    return text


def detect_sqli(path: str = "", query: str = "", body: str = "") -> SqliResult:
    text = _normalize(path, query, body)
    # Segunda variante sem comentários inline, para pegar evasões como UNION/**/SELECT
    variants = (text, _INLINE_COMMENT.sub(" ", text))

    found: dict[str, int] = {}
    for name, weight, rx in _SIGNATURES:
        if any(rx.search(v) for v in variants):
            found[name] = weight

    score = sum(found.values())
    return SqliResult(score >= SCORE_THRESHOLD, tuple(found), score)

## 4. Motor de correlação
O coração do projeto: mantém o estado por usuário/IP e escala a severidade conforme a cadeia se confirma.

In [4]:
"""Motor de correlação da kill chain.

Estágio 1: credencial corporativa vazada (feed de threat intel)
Estágio 2: login bem-sucedido com essa credencial, de origem anômala
Estágio 3: SQL Injection vindo da mesma sessão/IP do login suspeito

A severidade ESCALA conforme mais estágios são confirmados para o mesmo usuário.
O tempo usado é sempre o do evento (não o relógio da máquina), o que torna o
motor determinístico e fácil de testar/reprocessar.
"""
from __future__ import annotations

import logging
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta


log = logging.getLogger(__name__)


def _norm(user: str | None) -> str | None:
    return user.strip().lower() if user else None


@dataclass
class Chain:
    """Estado da cadeia de ataque de um usuário."""

    user: str
    leak: Event | None = None
    login: Event | None = None      # último login suspeito
    sqli_count: int = 0
    alerted: set = field(default_factory=set)   # evita alertas duplicados
    timeline: list = field(default_factory=list)

    def note(self, ts: datetime, text: str) -> None:
        self.timeline.append(f"{ts:%H:%M:%S} {text}")


RECO_LEAK = [
    "Forçar redefinição de senha da conta imediatamente",
    "Revogar sessões/tokens ativos e exigir MFA",
    "Revisar logins recentes da conta",
]
RECO_LOGIN = [
    "Encerrar as sessões do usuário e bloquear o IP de origem",
    "Forçar reset de senha e MFA",
    "Monitorar de perto toda atividade dessa conta",
]
RECO_SQLI_CHAIN = [
    "Bloquear o IP de origem no WAF/firewall AGORA",
    "Revogar sessões e redefinir a credencial do usuário",
    "Auditar o endpoint vulnerável e migrar para queries parametrizadas",
    "Acionar o plano de resposta a incidentes",
]
RECO_EXFIL = [
    "Acionar resposta a incidentes e preservar logs/evidências",
    "Identificar quais tabelas/dados foram expostos",
    "Avaliar obrigações de comunicação (LGPD) com o jurídico/DPO",
    "Rotacionar credenciais de banco e segredos da aplicação",
]
RECO_SQLI_ALONE = [
    "Verificar se o IP tem outras tentativas e considerar bloqueio",
    "Revisar o endpoint alvo (validação de entrada / queries parametrizadas)",
]


class Correlator:
    def __init__(self, config: Config | None = None):
        self.cfg = config or Config()
        self.chains: dict[str, Chain] = {}
        self.known_ips: dict[str, set] = defaultdict(set)        # baseline de IPs por usuário
        self.ip_sessions: dict[str, tuple] = {}                  # ip -> (usuário, quando logou)
        self.last_standalone: dict[str, datetime] = {}           # cooldown de SQLi solto

    # ------------------------------------------------------------------ API
    def process(self, event: Event) -> list[Alert]:
        """Recebe um evento e devolve zero ou mais alertas."""
        self._prune(event.timestamp)
        handlers = {
            EventType.CREDENTIAL_LEAK: self._on_leak,
            EventType.LOGIN: self._on_login,
            EventType.HTTP_REQUEST: self._on_http,
        }
        return handlers[event.type](event)

    # ------------------------------------------------------------- helpers
    def _prune(self, now: datetime) -> None:
        ttl = self.cfg.chain_window
        self.ip_sessions = {
            ip: (u, ts) for ip, (u, ts) in self.ip_sessions.items() if now - ts <= ttl
        }
        cd = self.cfg.standalone_cooldown
        self.last_standalone = {
            ip: ts for ip, ts in self.last_standalone.items() if now - ts <= cd
        }

    def _leak_active(self, chain: Chain | None, ts: datetime) -> bool:
        return bool(
            chain and chain.leak
            and timedelta(0) <= ts - chain.leak.timestamp <= self.cfg.leak_window
        )

    def _login_recent(self, chain: Chain, ts: datetime) -> bool:
        return bool(
            chain.login
            and timedelta(0) <= ts - chain.login.timestamp <= self.cfg.chain_window
        )

    def _user_from_ip(self, ip: str | None, ts: datetime) -> str | None:
        sess = self.ip_sessions.get(ip) if ip else None
        return sess[0] if sess else None

    # ------------------------------------------------------ estágio 1: leak
    def _on_leak(self, e: Event) -> list[Alert]:
        user = _norm(e.user)
        if not user:
            return []
        chain = self.chains.setdefault(user, Chain(user))
        # Vazamento antigo e expirado -> começa uma cadeia nova
        if chain.leak is not None and not self._leak_active(chain, e.timestamp):
            chain = self.chains[user] = Chain(user)
        if chain.leak is not None:  # mesmo vazamento reportado de novo: ignora
            chain.note(e.timestamp, "Vazamento repetido (ignorado)")
            return []

        chain.leak = e
        source = str(e.data.get("source", "fonte desconhecida"))
        chain.note(e.timestamp, f"Credencial vazada detectada ({source})")
        chain.alerted.add("leak")
        return [Alert(
            severity=Severity.MEDIUM,
            title="Credencial corporativa vazada (Infostealer)",
            stages=[1], user=user,
            details={
                "Fonte": source,
                "Família do malware": str(e.data.get("malware_family", "?")),
                "Host infectado": str(e.data.get("infected_host", "?")),
            },
            timeline=list(chain.timeline),
            recommendations=RECO_LEAK,
        )]

    # ------------------------------------------------------ estágio 2: login
    def _on_login(self, e: Event) -> list[Alert]:
        if not e.data.get("success", True):
            return []   # (melhoria futura: contar falhas -> credential stuffing)
        user, ip = _norm(e.user), e.src_ip
        if not user or not ip:
            return []

        chain = self.chains.get(user)
        leak_active = self._leak_active(chain, e.timestamp)
        new_ip = ip not in self.known_ips[user]
        foreign = bool(e.country) and e.country not in self.cfg.trusted_countries

        suspicious = (leak_active and new_ip) or (foreign and new_ip)
        if not suspicious:
            # Aprende o baseline apenas de logins "limpos"
            if not leak_active and not foreign:
                self.known_ips[user].add(ip)
            return []

        chain = self.chains.setdefault(user, Chain(user))
        chain.login = e
        self.ip_sessions[ip] = (user, e.timestamp)
        chain.note(e.timestamp, f"Login com sucesso de IP novo {ip} ({e.country or '?'})")

        key = f"login:{ip}"
        if key in chain.alerted:
            return []
        chain.alerted.add(key)

        details = {
            "User-Agent": str(e.data.get("user_agent", "?")),
            "Aplicação": str(e.data.get("app", "?")),
        }
        if leak_active:
            return [Alert(
                severity=Severity.HIGH,
                title="Login com credencial vazada",
                stages=[1, 2], user=user, src_ip=ip, country=e.country,
                details=details, timeline=list(chain.timeline),
                recommendations=RECO_LOGIN,
            )]
        return [Alert(
            severity=Severity.MEDIUM,
            title="Login anômalo (origem fora do padrão)",
            stages=[2], user=user, src_ip=ip, country=e.country,
            details=details, timeline=list(chain.timeline),
            recommendations=RECO_LOGIN,
        )]

    # -------------------------------------------------------- estágio 3: SQLi
    def _on_http(self, e: Event) -> list[Alert]:
        res = detect_sqli(e.data.get("path", ""), e.data.get("query", ""), e.data.get("body", ""))
        if not res.detected:
            return []

        user = _norm(e.user) or self._user_from_ip(e.src_ip, e.timestamp)
        chain = self.chains.get(user) if user else None
        same_session = (
            chain is not None and chain.login is not None
            and chain.login.src_ip == e.src_ip
            and self._login_recent(chain, e.timestamp)
        )
        if same_session:
            return self._on_chain_sqli(chain, e, res)
        return self._on_standalone_sqli(e, res, user)

    def _sqli_details(self, e: Event, res: SqliResult) -> dict[str, str]:
        return {
            "Alvo": str(e.data.get("path", "?")),
            "Assinaturas": ", ".join(res.signatures),
            "Status HTTP": str(e.data.get("status", "?")),
            "Bytes retornados": str(e.data.get("bytes_out", "?")),
        }

    def _looks_like_exfil(self, e: Event) -> bool:
        return (
            e.data.get("status") == 200
            and int(e.data.get("bytes_out", 0)) >= self.cfg.exfil_bytes_threshold
        )

    def _on_chain_sqli(self, chain: Chain, e: Event, res: SqliResult) -> list[Alert]:
        chain.sqli_count += 1
        leak_active = self._leak_active(chain, e.timestamp)
        exfil = self._looks_like_exfil(e)
        chain.note(e.timestamp, f"SQL Injection ({', '.join(res.signatures)}) em {e.data.get('path')}")
        if exfil:
            chain.note(e.timestamp, f"Resposta de {e.data.get('bytes_out')} bytes: possível exfiltração")

        common = dict(user=chain.user, src_ip=e.src_ip, country=e.country,
                      details=self._sqli_details(e, res), timeline=list(chain.timeline))
        alerts: list[Alert] = []

        if "sqli" not in chain.alerted:
            chain.alerted.add("sqli")
            if exfil:
                chain.alerted.add("exfil")
            if leak_active:
                alerts.append(Alert(
                    severity=Severity.CRITICAL,
                    title="CADEIA DE ATAQUE COMPLETA: vazamento → login → SQL Injection",
                    stages=[1, 2, 3], recommendations=RECO_SQLI_CHAIN, **common))
            else:
                alerts.append(Alert(
                    severity=Severity.HIGH,
                    title="SQL Injection após login anômalo",
                    stages=[2, 3], recommendations=RECO_SQLI_CHAIN, **common))
        elif exfil and "exfil" not in chain.alerted:
            chain.alerted.add("exfil")
            alerts.append(Alert(
                severity=Severity.CRITICAL,
                title="Possível EXFILTRAÇÃO de dados via SQL Injection",
                stages=[1, 2, 3] if leak_active else [2, 3],
                recommendations=RECO_EXFIL, **common))
        return alerts

    def _on_standalone_sqli(self, e: Event, res: SqliResult, user: str | None) -> list[Alert]:
        ip = e.src_ip or "?"
        if ip in self.last_standalone:      # cooldown contra spam de scanners
            return []
        self.last_standalone[ip] = e.timestamp
        return [Alert(
            severity=Severity.HIGH if user else Severity.MEDIUM,
            title="Tentativa de SQL Injection (sem correlação com vazamento)",
            stages=[3], user=user, src_ip=e.src_ip, country=e.country,
            details=self._sqli_details(e, res),
            timeline=[f"{e.timestamp:%H:%M:%S} SQL Injection em {e.data.get('path')}"],
            recommendations=RECO_SQLI_ALONE,
        )]

### Notificador do Telegram
Usa só a biblioteca padrão, com retry, tratamento de rate limit (HTTP 429) e modo dry-run.

In [5]:
"""Envio de alertas ao Telegram via Bot API (somente biblioteca padrão)."""
from __future__ import annotations

import html
import json
import logging
import re
import time
import urllib.error
import urllib.request


log = logging.getLogger(__name__)


def _esc(value: object) -> str:
    return html.escape(str(value), quote=False)


class TelegramNotifier:
    API_URL = "https://api.telegram.org/bot{token}/sendMessage"
    MAX_LEN = 4096          # limite do Telegram por mensagem
    MAX_TIMELINE = 12       # linhas da linha do tempo exibidas

    def __init__(self, token: str | None, chat_id: str | None, *,
                 dry_run: bool = False, retries: int = 3, timeout: float = 10.0):
        self.token, self.chat_id = token, chat_id
        # Sem credenciais, cai automaticamente para modo console
        self.dry_run = dry_run or not (token and chat_id)
        self.retries, self.timeout = retries, timeout
        if self.dry_run and not dry_run:
            log.warning("Telegram não configurado: alertas serão apenas impressos (dry-run).")

    # ---------------------------------------------------------- formatação
    @classmethod
    def format_alert(cls, a: Alert) -> str:
        lines = [f"{a.severity.emoji} <b>[{a.severity.label}] {_esc(a.title)}</b>", ""]
        if a.user:
            lines.append(f"👤 <b>Usuário:</b> <code>{_esc(a.user)}</code>")
        if a.src_ip:
            where = f" ({_esc(a.country)})" if a.country else ""
            lines.append(f"🌐 <b>IP de origem:</b> <code>{_esc(a.src_ip)}</code>{where}")
        for key, value in a.details.items():
            lines.append(f"• <b>{_esc(key)}:</b> {_esc(value)}")

        lines += ["", "🧩 <b>Cadeia de ataque</b>"]
        for n, name in STAGE_NAMES.items():
            lines.append(f"{'✅' if n in a.stages else '⬜'} {n}. {_esc(name)}")

        if a.timeline:
            tl = "\n".join(a.timeline[-cls.MAX_TIMELINE:])
            lines += ["", "🕒 <b>Linha do tempo (UTC)</b>", f"<pre>{_esc(tl)}</pre>"]
        if a.recommendations:
            lines += ["", "🛡️ <b>Ações recomendadas</b>"]
            lines += [f"{i}. {_esc(r)}" for i, r in enumerate(a.recommendations, 1)]
        return "\n".join(lines)[: cls.MAX_LEN]

    # -------------------------------------------------------------- envio
    def send(self, alert: Alert) -> bool:
        return self.send_text(self.format_alert(alert))

    def send_text(self, text: str) -> bool:
        if self.dry_run:
            self._print_dry_run(text)
            return True

        url = self.API_URL.format(token=self.token)   # nunca logar esta URL (contém o token)
        payload = json.dumps({
            "chat_id": self.chat_id, "text": text,
            "parse_mode": "HTML", "disable_web_page_preview": True,
        }).encode("utf-8")

        for attempt in range(1, self.retries + 1):
            wait = 2 ** attempt
            try:
                req = urllib.request.Request(
                    url, data=payload, method="POST",
                    headers={"Content-Type": "application/json"})
                with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                    body = json.loads(resp.read().decode("utf-8"))
                if body.get("ok"):
                    return True
                log.error("Telegram respondeu ok=false: %s", body.get("description"))
                return False
            except urllib.error.HTTPError as exc:
                detail = self._read_json(exc)
                if exc.code == 429:      # rate limit: respeita o retry_after
                    wait = detail.get("parameters", {}).get("retry_after", wait)
                elif 400 <= exc.code < 500:   # token/chat_id inválido, etc.: não adianta repetir
                    log.error("Telegram HTTP %s: %s", exc.code, detail.get("description"))
                    return False
                log.warning("Telegram HTTP %s (tentativa %d/%d)", exc.code, attempt, self.retries)
            except (urllib.error.URLError, TimeoutError, OSError) as exc:
                log.warning("Falha de rede ao enviar (tentativa %d/%d): %s", attempt, self.retries, exc)
            if attempt < self.retries:
                time.sleep(wait)

        log.error("Alerta NÃO entregue ao Telegram após %d tentativas.", self.retries)
        return False

    @staticmethod
    def _read_json(exc: urllib.error.HTTPError) -> dict:
        try:
            return json.loads(exc.read().decode("utf-8"))
        except Exception:
            return {}

    @staticmethod
    def _print_dry_run(text: str) -> None:
        plain = html.unescape(re.sub(r"</?(?:b|code|pre)>", "", text))
        bar = "─" * 64
        print(f"\n{bar}\n📨 [DRY-RUN] mensagem que seria enviada ao Telegram\n{bar}\n{plain}\n{bar}\n")

### Simulador de eventos
Quatro cenários: **A)** ataque completo · **B)** vazamento sem uso · **C)** scanner anônimo · **D)** login de outro país.

In [6]:
"""Simulador de eventos SINTÉTICOS (nenhum malware ou exploit real).

Gera tráfego normal + 4 cenários, usando IPs de documentação (RFC 5737)
e um domínio fictício:

  A) Ataque completo contra maria: vazamento -> login anômalo -> SQLi -> exfiltração
  B) Vazamento sem uso (ana)                         -> só estágio 1
  C) Scanner anônimo tentando SQLi                   -> só estágio 3 (sem correlação)
  D) Login legítimo, mas de outro país (carlos)      -> só estágio 2 (falso positivo provável)
"""
from __future__ import annotations

import logging
import time
from datetime import datetime, timedelta, timezone
from typing import Iterator


log = logging.getLogger(__name__)

CORP = "empresa-exemplo.com.br"
MARIA, JOAO, ANA, CARLOS = (f"{n}@{CORP}" for n in
                            ("maria.silva", "joao.souza", "ana.lima", "carlos.dias"))
OFFICE_IPS = {MARIA: "198.51.100.10", JOAO: "198.51.100.11",
              ANA: "198.51.100.12", CARLOS: "198.51.100.13"}
ATTACKER_IP = "203.0.113.77"   # saída de VPN/proxy no exterior
SCANNER_IP = "192.0.2.99"
TRAVEL_IP = "203.0.113.20"


def _now() -> datetime:
    return datetime.now(timezone.utc)


def _leak(user: str) -> Event:
    return Event(EventType.CREDENTIAL_LEAK, _now(), user=user, data={
        "source": "Infostealer log à venda em fórum clandestino",
        "malware_family": "RedLine (simulado)",
        "infected_host": "DESKTOP-4F2K9",
        "leaked_at": (_now() - timedelta(days=3)).isoformat(),
    })


def _login(user: str, ip: str, country: str,
           ua: str = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/118.0") -> Event:
    return Event(EventType.LOGIN, _now(), user=user, src_ip=ip, country=country,
                 data={"success": True, "user_agent": ua, "app": "VPN corporativa"})


def _http(ip: str, path: str, *, query: str = "", user: str | None = None,
          country: str = "BR", status: int = 200, bytes_out: int = 1_500) -> Event:
    return Event(EventType.HTTP_REQUEST, _now(), user=user, src_ip=ip, country=country,
                 data={"method": "GET", "path": path, "query": query,
                       "status": status, "bytes_out": bytes_out})


def simulate(delay: float = 1.0) -> Iterator[Event]:
    """Gera eventos em 'tempo real'. `delay` = segundos entre eventos (0 = o mais rápido possível)."""

    def pace(factor: float = 1.0) -> None:
        if delay > 0:
            time.sleep(delay * factor)

    log.info("=== Fase 1: tráfego normal (baseline) ===")
    for user, ip in OFFICE_IPS.items():
        yield _login(user, ip, "BR"); pace(0.5)
    for user, ip in OFFICE_IPS.items():
        yield _http(ip, "/dashboard", user=user); pace(0.5)
    yield _http(OFFICE_IPS[JOAO], "/search", query="q=notebook+gamer", user=JOAO); pace()

    log.info("=== Cenário B: vazamento sem uso (ana) ===")
    yield _leak(ANA); pace(2)

    log.info("=== Cenário C: scanner anônimo tentando SQLi ===")
    for _ in range(3):   # 3 tentativas -> só 1 alerta (cooldown)
        yield _http(SCANNER_IP, "/api/products", query="id=1 UNION SELECT NULL,NULL--",
                    country="DE", status=403, bytes_out=300); pace(0.3)
    pace(1.5)

    log.info("=== Cenário D: carlos viajando (login de outro país, sem vazamento) ===")
    yield _login(CARLOS, TRAVEL_IP, "PT"); pace(2)

    log.info("=== Cenário A: ATAQUE COMPLETO contra maria ===")
    yield _leak(MARIA); pace(2)
    ua = "Mozilla/5.0 (X11; Linux x86_64; rv:118.0) Gecko/20100101 Firefox/118.0"
    yield _login(MARIA, ATTACKER_IP, "NL", ua=ua); pace(2)
    yield _http(ATTACKER_IP, "/dashboard", user=MARIA, country="NL"); pace()
    yield _http(ATTACKER_IP, "/api/orders", query="id=1", user=MARIA, country="NL"); pace()
    yield _http(ATTACKER_IP, "/api/orders", query="id=1' OR '1'='1' --",
                user=MARIA, country="NL", bytes_out=3_200); pace(2)
    yield _http(ATTACKER_IP, "/api/orders",
                query="id=1%20UNION%20SELECT%20username%2Cpassword%20FROM%20users--",
                user=MARIA, country="NL", bytes_out=2_400_000)
    log.info("=== Fim da simulação ===")

## 5. Credenciais do Telegram

1. No Telegram, abra o **@BotFather**, envie `/newbot` e copie o **token**.
2. Envie qualquer mensagem para o seu bot (ou adicione-o a um grupo e escreva algo).
3. Abra `https://api.telegram.org/bot<SEU_TOKEN>/getUpdates` no navegador e copie o `chat.id`
   (em grupos costuma ser negativo).
4. No Colab, clique no ícone **🔑 Secrets** (barra lateral esquerda), crie `TELEGRAM_BOT_TOKEN` e
   `TELEGRAM_CHAT_ID` e ative **Notebook access**.

> 🔒 Nunca cole o token direto no código nem compartilhe o notebook com ele preenchido.
> Se vazar, revogue com `/revoke` no BotFather.

In [7]:
import getpass
import os


def _get_secret(name):
    """Procura em: Secrets do Colab (ícone 🔑 na barra lateral) -> variável de ambiente."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name) or None


TELEGRAM_BOT_TOKEN = _get_secret("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = _get_secret("TELEGRAM_CHAT_ID")

if not (TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID):
    print("Secrets do Colab não encontrados. Cole os dados abaixo, ou deixe em branco "
          "e pressione Enter para usar o modo DRY-RUN (alertas só aparecem na tela).")
    try:
        TELEGRAM_BOT_TOKEN = getpass.getpass("Token do bot: ").strip() or None
        TELEGRAM_CHAT_ID = input("chat_id: ").strip() if TELEGRAM_BOT_TOKEN else None
        TELEGRAM_CHAT_ID = TELEGRAM_CHAT_ID or None
    except EOFError:
        TELEGRAM_BOT_TOKEN = TELEGRAM_CHAT_ID = None

cfg = Config(telegram_token=TELEGRAM_BOT_TOKEN, telegram_chat_id=TELEGRAM_CHAT_ID)

if cfg.telegram_token and cfg.telegram_chat_id:
    print("✅ Telegram configurado.")   # o token nunca é exibido
else:
    print("ℹ️ Telegram NÃO configurado: rodando em modo DRY-RUN.")

Secrets do Colab não encontrados. Cole os dados abaixo, ou deixe em branco e pressione Enter para usar o modo DRY-RUN (alertas só aparecem na tela).
Token do bot: ··········
chat_id: 5785545918
✅ Telegram configurado.


## 6. Teste de conexão com o Telegram *(opcional)*

In [8]:
notifier = TelegramNotifier(cfg.telegram_token, cfg.telegram_chat_id)
ok = notifier.send_text("✅ <b>Teste</b>: o correlacionador de ameaças está conectado ao Telegram.")
print("Resultado:", "OK ✅" if ok else "FALHOU ❌ (veja a mensagem de erro acima)")

Resultado: OK ✅


## 7. ▶️ Executar a simulação
Os eventos chegam em "tempo real" (um a cada `DELAY` segundos) e cada alerta é enviado na hora.

In [9]:
import logging
import sys

# ===================== PARÂMETROS =====================
DELAY = 1.0             # segundos entre eventos simulados (0 = instantâneo)
FORCE_DRY_RUN = False   # True = só imprime na tela, mesmo com o Telegram configurado
# ======================================================

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True,
                    format="%(asctime)s %(levelname)-7s %(message)s", datefmt="%H:%M:%S")

notifier = TelegramNotifier(cfg.telegram_token, cfg.telegram_chat_id, dry_run=FORCE_DRY_RUN)
correlator = Correlator(cfg)      # estado novo a cada execução
sent = failed = 0

for event in simulate(delay=DELAY):
    log.info("evento  %s", event.summary())
    for alert in correlator.process(event):
        log.warning("ALERTA  [%s] %s", alert.severity.label, alert.title)
        if notifier.send(alert):
            sent += 1
        else:
            failed += 1

print(f"\nResumo: {sent} alertas entregues, {failed} falhas.")

20:32:34 INFO    === Fase 1: tráfego normal (baseline) ===
20:32:34 INFO    evento  [login] user=maria.silva@empresa-exemplo.com.br ip=198.51.100.10 país=BR
20:32:34 INFO    evento  [login] user=joao.souza@empresa-exemplo.com.br ip=198.51.100.11 país=BR
20:32:35 INFO    evento  [login] user=ana.lima@empresa-exemplo.com.br ip=198.51.100.12 país=BR
20:32:35 INFO    evento  [login] user=carlos.dias@empresa-exemplo.com.br ip=198.51.100.13 país=BR
20:32:36 INFO    evento  [http_request] user=maria.silva@empresa-exemplo.com.br ip=198.51.100.10 país=BR /dashboard?
20:32:36 INFO    evento  [http_request] user=joao.souza@empresa-exemplo.com.br ip=198.51.100.11 país=BR /dashboard?
20:32:37 INFO    evento  [http_request] user=ana.lima@empresa-exemplo.com.br ip=198.51.100.12 país=BR /dashboard?
20:32:37 INFO    evento  [http_request] user=carlos.dias@empresa-exemplo.com.br ip=198.51.100.13 país=BR /dashboard?
20:32:38 INFO    evento  [http_request] user=joao.souza@empresa-exemplo.com.br ip=198.51.

## 8. Testes automatizados
16 testes cobrindo detector, escalonamento, janelas de tempo, deduplicação, exfiltração e o notificador.

In [10]:
import io
import json
import unittest
import urllib.error
from datetime import datetime, timedelta, timezone
from unittest.mock import patch


T0 = datetime(2026, 11, 3, 12, 0, tzinfo=timezone.utc)
USER = "maria.silva@empresa-exemplo.com.br"
OFFICE, ATTACKER, SCANNER = "198.51.100.10", "203.0.113.77", "192.0.2.99"


def leak(at, user=USER):
    return Event(EventType.CREDENTIAL_LEAK, at, user=user, data={"source": "teste"})


def login(at, ip=OFFICE, country="BR", user=USER):
    return Event(EventType.LOGIN, at, user=user, src_ip=ip, country=country, data={"success": True})


def http(at, query="id=1", ip=OFFICE, user=USER, bytes_out=1000, status=200):
    return Event(EventType.HTTP_REQUEST, at, user=user, src_ip=ip, country="BR",
                 data={"path": "/api/orders", "query": query, "status": status, "bytes_out": bytes_out})


def sec(s):
    return T0 + timedelta(seconds=s)


class SqliDetectorTests(unittest.TestCase):
    def test_detects_known_patterns(self):
        cases = [
            "id=1' OR '1'='1' --",
            "id=1%20UNION%20SELECT%20username%2Cpassword%20FROM%20users--",
            "id=1; DROP TABLE users",
            "id=1 AND SLEEP(5)",
            "id=1%2527%2520OR%25201%253D1",          # dupla codificação
            "id=1 UNION/**/SELECT null,null",         # evasão com comentário inline
        ]
        for q in cases:
            with self.subTest(q=q):
                self.assertTrue(detect_sqli("/api", q).detected)

    def test_ignores_normal_traffic(self):
        for path, q in [("/search", "q=notebook+gamer"), ("/api/orders", "id=123"),
                        ("/users", "name=O'Reilly"), ("/blog", "title=how to select a union plan")]:
            with self.subTest(q=q):
                self.assertFalse(detect_sqli(path, q).detected)


class CorrelatorTests(unittest.TestCase):
    def setUp(self):
        self.c = Correlator(Config())
        self.c.process(login(T0))   # baseline: IP do escritório

    def test_leak_alone_is_medium(self):
        alerts = self.c.process(leak(sec(10)))
        self.assertEqual([(a.severity, a.stages) for a in alerts], [(Severity.MEDIUM, [1])])

    def test_duplicate_leak_is_ignored(self):
        self.c.process(leak(sec(10)))
        self.assertEqual(self.c.process(leak(sec(20))), [])

    def test_full_chain_escalates_to_critical(self):
        got = []
        got += self.c.process(leak(sec(10)))
        got += self.c.process(login(sec(60), ATTACKER, "NL"))
        got += self.c.process(http(sec(90), "id=1' OR '1'='1' --", ip=ATTACKER))
        self.assertEqual([a.severity for a in got],
                         [Severity.MEDIUM, Severity.HIGH, Severity.CRITICAL])
        self.assertEqual(got[-1].stages, [1, 2, 3])
        self.assertEqual(got[-1].src_ip, ATTACKER)
        self.assertGreaterEqual(len(got[-1].timeline), 3)

    def test_sqli_correlated_by_ip_when_request_is_unauthenticated(self):
        self.c.process(leak(sec(10)))
        self.c.process(login(sec(60), ATTACKER, "NL"))
        alerts = self.c.process(http(sec(90), "id=1 UNION SELECT 1,2--", ip=ATTACKER, user=None))
        self.assertEqual(alerts[0].stages, [1, 2, 3])

    def test_login_from_known_ip_with_leak_is_not_stage2(self):
        self.c.process(leak(sec(10)))
        self.assertEqual(self.c.process(login(sec(60), OFFICE, "BR")), [])

    def test_foreign_login_without_leak_is_medium_stage2(self):
        alerts = self.c.process(login(sec(60), ATTACKER, "NL"))
        self.assertEqual([(a.severity, a.stages) for a in alerts], [(Severity.MEDIUM, [2])])

    def test_expired_leak_does_not_count(self):
        self.c.process(leak(sec(10)))
        later = T0 + timedelta(days=31)
        alerts = self.c.process(login(later, ATTACKER, "NL"))
        self.assertEqual(alerts[0].stages, [2])
        self.assertEqual(alerts[0].severity, Severity.MEDIUM)

    def test_sqli_without_context_is_medium_and_rate_limited(self):
        ev = lambda s: http(sec(s), "id=1 UNION SELECT 1--", ip=SCANNER, user=None)
        first = self.c.process(ev(10))
        self.assertEqual([(a.severity, a.stages) for a in first], [(Severity.MEDIUM, [3])])
        self.assertEqual(self.c.process(ev(20)), [])           # cooldown
        self.assertEqual(len(self.c.process(ev(200))), 1)      # depois do cooldown

    def test_sqli_after_chain_window_is_not_correlated(self):
        self.c.process(leak(sec(10)))
        self.c.process(login(sec(60), ATTACKER, "NL"))
        alerts = self.c.process(http(sec(60 + 3 * 3600), "id=1 UNION SELECT 1--", ip=ATTACKER))
        self.assertEqual(alerts[0].stages, [3])

    def test_exfiltration_generates_second_critical_alert(self):
        self.c.process(leak(sec(10)))
        self.c.process(login(sec(60), ATTACKER, "NL"))
        first = self.c.process(http(sec(90), "id=1' OR '1'='1' --", ip=ATTACKER))
        second = self.c.process(http(sec(120), "id=1 UNION SELECT a,b FROM t--",
                                     ip=ATTACKER, bytes_out=2_000_000))
        self.assertEqual(len(first), 1)
        self.assertEqual(second[0].severity, Severity.CRITICAL)
        self.assertIn("EXFILTRAÇÃO", second[0].title)
        # e não repete
        self.assertEqual(self.c.process(http(sec(130), "id=1 UNION SELECT a,b FROM t--",
                                             ip=ATTACKER, bytes_out=2_000_000)), [])


class FakeResponse:
    def __init__(self, body: bytes):
        self._body = body
    def read(self):
        return self._body
    def __enter__(self):
        return self
    def __exit__(self, *exc):
        return False


class NotifierTests(unittest.TestCase):
    def alert(self):
        return Alert(Severity.CRITICAL, "Teste <script>", [1, 2, 3], user=USER, src_ip=ATTACKER,
                     country="NL", details={"Alvo": "/api/orders"},
                     timeline=["12:00:00 evento & teste"], recommendations=["Bloquear IP"])

    def test_format_escapes_html_and_shows_stages(self):
        text = TelegramNotifier.format_alert(self.alert())
        self.assertIn("&lt;script&gt;", text)
        self.assertIn("evento &amp; teste", text)
        self.assertEqual(text.count("✅"), 3)
        self.assertLessEqual(len(text), 4096)

    def test_dry_run_when_not_configured(self):
        self.assertTrue(TelegramNotifier(None, None).dry_run)

    @patch("time.sleep")
    @patch("urllib.request.urlopen")
    def test_retries_on_5xx_then_succeeds(self, urlopen, _sleep):
        err = urllib.error.HTTPError("u", 502, "Bad Gateway", {}, io.BytesIO(b"{}"))
        urlopen.side_effect = [err, FakeResponse(json.dumps({"ok": True}).encode())]
        n = TelegramNotifier("123:TOKEN", "42")
        self.assertTrue(n.send(self.alert()))
        self.assertEqual(urlopen.call_count, 2)

    @patch("time.sleep")
    @patch("urllib.request.urlopen")
    def test_does_not_retry_on_401(self, urlopen, _sleep):
        body = io.BytesIO(json.dumps({"ok": False, "description": "Unauthorized"}).encode())
        urlopen.side_effect = urllib.error.HTTPError("u", 401, "Unauthorized", {}, body)
        self.assertFalse(TelegramNotifier("bad", "42").send(self.alert()))
        self.assertEqual(urlopen.call_count, 1)


unittest.main(argv=["colab"], exit=False, verbosity=2)

test_duplicate_leak_is_ignored (__main__.CorrelatorTests.test_duplicate_leak_is_ignored) ... ok
test_exfiltration_generates_second_critical_alert (__main__.CorrelatorTests.test_exfiltration_generates_second_critical_alert) ... ok
test_expired_leak_does_not_count (__main__.CorrelatorTests.test_expired_leak_does_not_count) ... ok
test_foreign_login_without_leak_is_medium_stage2 (__main__.CorrelatorTests.test_foreign_login_without_leak_is_medium_stage2) ... ok
test_full_chain_escalates_to_critical (__main__.CorrelatorTests.test_full_chain_escalates_to_critical) ... ok
test_leak_alone_is_medium (__main__.CorrelatorTests.test_leak_alone_is_medium) ... ok
test_login_from_known_ip_with_leak_is_not_stage2 (__main__.CorrelatorTests.test_login_from_known_ip_with_leak_is_not_stage2) ... ok
test_sqli_after_chain_window_is_not_correlated (__main__.CorrelatorTests.test_sqli_after_chain_window_is_not_correlated) ... ok
test_sqli_correlated_by_ip_when_request_is_unauthenticated (__main__.CorrelatorTes

20:35:08 ERROR   Telegram HTTP 401: Unauthorized


ok
test_dry_run_when_not_configured (__main__.NotifierTests.test_dry_run_when_not_configured) ... 

20:35:08 WARNING Telegram não configurado: alertas serão apenas impressos (dry-run).


ok
test_format_escapes_html_and_shows_stages (__main__.NotifierTests.test_format_escapes_html_and_shows_stages) ... ok
test_retries_on_5xx_then_succeeds (__main__.NotifierTests.test_retries_on_5xx_then_succeeds) ... 

20:35:08 WARNING Telegram HTTP 502 (tentativa 1/3)


ok
test_detects_known_patterns (__main__.SqliDetectorTests.test_detects_known_patterns) ... ok
test_ignores_normal_traffic (__main__.SqliDetectorTests.test_ignores_normal_traffic) ... ok

----------------------------------------------------------------------
Ran 16 tests in 0.041s

OK


## 9. Experimente: eventos escritos por você
Monte sua própria sequência de eventos e veja a correlação (aqui em dry-run; mude para `dry_run=False` para enviar ao Telegram).

In [11]:
# Exemplo: alimentando o correlacionador com eventos escritos à mão
from datetime import datetime, timedelta, timezone

USER = "paulo.reis@empresa-exemplo.com.br"
t = datetime.now(timezone.utc)

eventos = [
    # baseline: Paulo costuma logar do IP do escritório
    Event(EventType.LOGIN, t, user=USER, src_ip="198.51.100.50", country="BR", data={"success": True}),
    # estágio 1: a credencial dele aparece num log de infostealer
    Event(EventType.CREDENTIAL_LEAK, t + timedelta(minutes=5), user=USER,
          data={"source": "Telegram channel de logs", "malware_family": "Lumma (simulado)"}),
    # estágio 2: login de um IP novo, em outro país
    Event(EventType.LOGIN, t + timedelta(minutes=9), user=USER, src_ip="203.0.113.99",
          country="NL", data={"success": True, "user_agent": "Firefox/118"}),
    # estágio 3: SQL Injection a partir desse mesmo IP
    Event(EventType.HTTP_REQUEST, t + timedelta(minutes=12), user=USER, src_ip="203.0.113.99",
          country="NL", data={"path": "/api/clientes", "query": "id=1 UNION SELECT cpf,senha FROM usuarios--",
                              "status": 200, "bytes_out": 1_800_000}),
]

meu_correlator = Correlator(cfg)
meu_notifier = TelegramNotifier(cfg.telegram_token, cfg.telegram_chat_id, dry_run=True)  # dry_run=False para enviar de verdade
for ev in eventos:
    for alert in meu_correlator.process(ev):
        meu_notifier.send(alert)


────────────────────────────────────────────────────────────────
📨 [DRY-RUN] mensagem que seria enviada ao Telegram
────────────────────────────────────────────────────────────────
🟡 [MÉDIO] Credencial corporativa vazada (Infostealer)

👤 Usuário: paulo.reis@empresa-exemplo.com.br
• Fonte: Telegram channel de logs
• Família do malware: Lumma (simulado)
• Host infectado: ?

🧩 Cadeia de ataque
✅ 1. Credencial vazada (Infostealer)
⬜ 2. Login com credencial legítima
⬜ 3. SQL Injection

🕒 Linha do tempo (UTC)
20:40:37 Credencial vazada detectada (Telegram channel de logs)

🛡️ Ações recomendadas
1. Forçar redefinição de senha da conta imediatamente
2. Revogar sessões/tokens ativos e exigir MFA
3. Revisar logins recentes da conta
────────────────────────────────────────────────────────────────


────────────────────────────────────────────────────────────────
📨 [DRY-RUN] mensagem que seria enviada ao Telegram
────────────────────────────────────────────────────────────────
🟠 [ALTO] Login com 

#10. Arquitetura de Produção

In [14]:
!pip install geoip2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geoip2]


In [17]:
"""
Módulo de Produção corrigido: Importação de dataclasses, validação de GeoIP e SQLite.
"""
from __future__ import annotations

import sqlite3
import queue
import threading
import time
import re
import logging
import dataclasses  # <-- CORREÇÃO: Importação necessária para dataclasses.replace
from datetime import datetime, timezone, timedelta
from typing import Iterator, Any

log = logging.getLogger(__name__)

# Tente importar bibliotecas opcionais de produção (caso não instaladas, fallback elegante)
try:
    import geoip2.database
    HAS_GEOIP = True
except ImportError:
    HAS_GEOIP = False

try:
    from elasticsearch import Elasticsearch
    HAS_ES = True
except ImportError:
    HAS_ES = False


# =====================================================================
# 1. GEOIP REAL (GeoLite2)
# =====================================================================
class GeoIPEnricher:
    """Enriquece o país do IP usando a base MaxMind GeoLite2 City/Country."""
    def __init__(self, db_path: str = "GeoLite2-Country.mmdb"):
        self.db_path = db_path
        self.reader = None
        if HAS_GEOIP and db_path:  # Só tenta abrir se houver um caminho válido
            try:
                self.reader = geoip2.database.Reader(db_path)
                log.info("Base GeoIP GeoLite2 carregada com sucesso de %s", db_path)
            except Exception as exc:
                log.warning("Não foi possível carregar a base GeoIP (%s). Usando fallback IP vazio.", exc)

    def get_country(self, ip: str) -> str | None:
        if not self.reader or not ip:
            return None
        # Ignora IPs locais/privados ou de documentação (RFC 5737, RFC 1918)
        if ip.startswith(("10.", "192.168.", "127.", "198.51.100.", "203.0.113.", "192.0.2.")):
            return "BR"  # ou None para IPs locais
        try:
            response = self.reader.country(ip)
            return response.country.iso_code
        except Exception:
            return None


# =====================================================================
# 2. PERSISTÊNCIA DE ESTADO COM SQLITE
# =====================================================================
class SQLiteStateStore:
    """Persiste o baseline de IPs conhecidos e sessões para sobreviver a restarts."""
    def __init__(self, db_path: str = "threat_correlator_state.db"):
        self.db_path = db_path
        self._init_db()

    def _get_conn(self):
        return sqlite3.connect(self.db_path)

    def _init_db(self) -> None:
        with self._get_conn() as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS known_ips (
                    user TEXT,
                    ip TEXT,
                    PRIMARY KEY (user, ip)
                )
            """)
            conn.execute("""
                CREATE TABLE IF NOT EXISTS ip_sessions (
                    ip TEXT PRIMARY KEY,
                    user TEXT,
                    timestamp TEXT
                )
            """)
            conn.execute("""
                CREATE TABLE IF NOT EXISTS failed_logins (
                    ip TEXT,
                    timestamp TEXT
                )
            """)
            conn.commit()

    def add_known_ip(self, user: str, ip: str) -> None:
        with self._get_conn() as conn:
            conn.execute("INSERT OR IGNORE INTO known_ips (user, ip) VALUES (?, ?)", (user, ip))
            conn.commit()

    def is_known_ip(self, user: str, ip: str) -> bool:
        with self._get_conn() as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT 1 FROM known_ips WHERE user = ? AND ip = ?", (user, ip))
            return cursor.fetchone() is not None

    def save_session(self, ip: str, user: str, ts: datetime) -> None:
        with self._get_conn() as conn:
            conn.execute("INSERT OR REPLACE INTO ip_sessions (ip, user, timestamp) VALUES (?, ?, ?)",
                         (ip, user, ts.isoformat()))
            conn.commit()

    def get_session(self, ip: str) -> tuple[str | None, datetime | None]:
        with self._get_conn() as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT user, timestamp FROM ip_sessions WHERE ip = ?", (ip,))
            row = cursor.fetchone()
            if row:
                return row[0], datetime.fromisoformat(row[1])
            return None, None

    def record_failed_login(self, ip: str, ts: datetime) -> None:
        with self._get_conn() as conn:
            conn.execute("INSERT INTO failed_logins (ip, timestamp) VALUES (?, ?)", (ip, ts.isoformat()))
            conn.commit()

    def count_recent_failures(self, ip: str, window: timedelta, now: datetime) -> int:
        limit_time = (now - window).isoformat()
        with self._get_conn() as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT COUNT(*) FROM failed_logins WHERE ip = ? AND timestamp >= ?", (ip, limit_time))
            res = cursor.fetchone()
            return res[0] if res else 0


# =====================================================================
# 3. CORRELAÇÃO RETROATIVA & CREDENTIAL STUFFING
# =====================================================================
class ProductionCorrelator(Correlator):
    """Extensão do correlator base com SQLite, Correlação Retroativa e Stuffing."""
    def __init__(self, config: Config | None = None, db_path: str = "threat_state.db", geoip_path: str = ""):
        super().__init__(config)
        self.store = SQLiteStateStore(db_path)
        self.geoip = GeoIPEnricher(geoip_path) if geoip_path else GeoIPEnricher("GeoLite2-Country.mmdb")
        self.stuffing_window = timedelta(minutes=5)
        self.stuffing_threshold = 5

    def process(self, event: Event) -> list[Alert]:
        if not event.country and event.src_ip:
            country_code = self.geoip.get_country(event.src_ip)
            if country_code:
                event = dataclasses.replace(event, country=country_code)

        alerts = super().process(event)

        if event.type == EventType.LOGIN and not event.data.get("success", True):
            ip = event.src_ip
            if ip:
                self.store.record_failed_login(ip, event.timestamp)
                failures = self.store.count_recent_failures(ip, self.stuffing_window, event.timestamp)
                if failures >= self.stuffing_threshold:
                    alerts.append(Alert(
                        severity=Severity.HIGH,
                        title="Possível ataque de Credential Stuffing / Brute-force",
                        stages=[], src_ip=ip, country=event.country,
                        details={"Tentativas falhas": str(failures), "Janela": "5 minutos"},
                        recommendations=["Bloquear IP no WAF/Firewall", "Ativar rate limiting no endpoint de login"]
                    ))
        return alerts

    def _on_leak(self, e: Event) -> list[Alert]:
        alerts = super()._on_leak(e)
        user = _norm(e.user)
        if not user:
            return alerts

        chain = self.chains.get(user)
        if chain and chain.login and not chain.leak:
            dt_diff = e.timestamp - chain.login.timestamp
            if timedelta(0) <= dt_diff <= self.cfg.leak_window:
                chain.leak = e
                chain.note(e.timestamp, f"Correlação retroativa: vazamento associado ao login anterior de {chain.login.src_ip}")
                alerts.append(Alert(
                    severity=Severity.HIGH,
                    title="Correlação Retroativa: Login anômalo seguido de vazamento",
                    stages=[1, 2], user=user, src_ip=chain.login.src_ip, country=chain.login.country,
                    details={"Nota": "O login ocorreu antes do alerta de infostealer ser publicado na Threat Intel."},
                    timeline=list(chain.timeline),
                    recommendations=RECO_LOGIN
                ))
        return alerts


# =====================================================================
# 4. FILA + WORKER ASSÍNCRONO
# =====================================================================
class AsyncTelegramDispatcher:
    """Enfileira alertas e utiliza um Worker thread-safe para envio assíncrono ao Telegram."""
    def __init__(self, notifier: TelegramNotifier):
        self.notifier = notifier
        self.queue: queue.Queue[Alert | None] = queue.Queue()
        self.worker_thread = threading.Thread(target=self._worker_loop, daemon=True)
        self.running = True
        self.worker_thread.start()

    def enqueue(self, alert: Alert) -> None:
        self.queue.put(alert)

    def _worker_loop(self) -> None:
        while self.running:
            try:
                alert = self.queue.get(timeout=1.0)
                if alert is None:
                    break
                self.notifier.send(alert)
                self.queue.task_done()
            except queue.Empty:
                continue
            except Exception as exc:
                log.error("Erro no worker de envio ao Telegram: %s", exc)

    def stop(self) -> None:
        self.running = False
        self.queue.put(None)
        self.worker_thread.join(timeout=3.0)


# =====================================================================
# 5. LEITORES REAIS (ADAPTERS)
# =====================================================================
class NginxLogTailer:
    """Exemplo de leitor de logs do Nginx / WAF via Regex em arquivos de log."""
    LOG_RX = re.compile(
        r'(?P<ip>[\d\.]+)\s+\S+\s+\S+\s+\[(?P<timestamp>[^\]]+)\]\s+"(?P<method>\w+)\s+(?P<uri>[^\s]+)\s+[^\"]+"\s+(?P<status>\d+)\s+(?P<bytes>\d+)'
    )

    @classmethod
    def parse_line(cls, line: str) -> Event | None:
        match = cls.LOG_RX.match(line)
        if not match:
            return None
        data = match.groupdict()
        path_query = data["uri"].split("?", 1)
        path = path_query[0]
        query = path_query[1] if len(path_query) > 1 else ""

        return Event(
            type=EventType.HTTP_REQUEST,
            timestamp=datetime.now(timezone.utc),
            src_ip=data["ip"],
            data={
                "method": data["method"],
                "path": path,
                "query": query,
                "status": int(data["status"]),
                "bytes_out": int(data["bytes"])
            }
        )

#11. Instanciar o ProductionCorrelator e o despachante assíncrono AsyncTelegramDispatcher

In [20]:
pip install maxminddb

In [21]:
pip install geoip2

In [23]:
# Inicialização do motor de produção (sem forçar o caminho GeoIP se não possuir a base .mmdb)
prod_correlator = ProductionCorrelator(cfg, db_path="/content/producao_siem.db", geoip_path="/content/GeoLite2-Country-Test.mmdb")
notifier = TelegramNotifier(cfg.telegram_token, cfg.telegram_chat_id, dry_run=True)
#
async_dispatcher = AsyncTelegramDispatcher(notifier)

# Exemplo de processamento de um log real do Nginx capturado:
linha_nginx = '203.0.113.77 - - [03/Nov/2026:12:00:00 +0000] "GET /api/orders?id=1%20UNION%20SELECT%201,2-- HTTP/1.1" 200 2500000'
evento_real = NginxLogTailer.parse_line(linha_nginx)

if evento_real:
    print(f"Evento parseado do Nginx: {evento_real.summary()}")
    alertas = prod_correlator.process(evento_real)
    for alerta in alertas:
        async_dispatcher.enqueue(alerta)

# Aguarda o envio assíncrono na fila
time.sleep(2)
async_dispatcher.stop()
print("Processamento de produção finalizado com sucesso!")

21:34:07 INFO    Base GeoIP GeoLite2 carregada com sucesso de /content/GeoLite2-Country-Test.mmdb
Evento parseado do Nginx: [http_request] ip=203.0.113.77 /api/orders?id=1%20UNION%20SELECT%201,2--

────────────────────────────────────────────────────────────────
📨 [DRY-RUN] mensagem que seria enviada ao Telegram
────────────────────────────────────────────────────────────────
🟡 [MÉDIO] Tentativa de SQL Injection (sem correlação com vazamento)

🌐 IP de origem: 203.0.113.77 (BR)
• Alvo: /api/orders
• Assinaturas: UNION SELECT
• Status HTTP: 200
• Bytes retornados: 2500000

🧩 Cadeia de ataque
⬜ 1. Credencial vazada (Infostealer)
⬜ 2. Login com credencial legítima
✅ 3. SQL Injection

🕒 Linha do tempo (UTC)
21:34:07 SQL Injection em /api/orders

🛡️ Ações recomendadas
1. Verificar se o IP tem outras tentativas e considerar bloqueio
2. Revisar o endpoint alvo (validação de entrada / queries parametrizadas)
────────────────────────────────────────────────────────────────

Processamento de prod